# 🧠 Alzheimer's MRI Stage Classifier
### CNN-based Multi-class Classification with Transfer Learning + Grad-CAM Explainability

**Dataset:** Preprocessed Alzheimer's Disease MRI Dataset  
**Classes:** Mild Dementia | Moderate Dementia | Non Demented | Very Mild Dementia  
**Model:** EfficientNetB0 (Transfer Learning)  
**Unique Feature:** Grad-CAM heatmaps showing which brain regions the model focuses on


## 1. Import Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam

from sklearn.metrics import classification_report, confusion_matrix

print(f'TensorFlow version: {tf.__version__}')
print('All libraries loaded successfully!')

## 2. Dataset Configuration

In [ ]:
# ============================================================
# UPDATE THIS PATH to where you extracted the dataset
# ============================================================
BASE_DIR = './alziemer_dataset'   # <-- change if needed

TRAIN_DIR = os.path.join(BASE_DIR, 'train')
VAL_DIR   = os.path.join(BASE_DIR, 'val')
TEST_DIR  = os.path.join(BASE_DIR, 'test')

CLASS_NAMES = sorted(os.listdir(TRAIN_DIR))
NUM_CLASSES = len(CLASS_NAMES)

IMG_SIZE    = (224, 224)
BATCH_SIZE  = 32
EPOCHS      = 20

print('Classes found:', CLASS_NAMES)
print('Number of classes:', NUM_CLASSES)

## 3. Explore Dataset — Class Distribution

In [ ]:
def count_images(directory):
    counts = {}
    for cls in CLASS_NAMES:
        path = os.path.join(directory, cls)
        counts[cls] = len(os.listdir(path))
    return counts

train_counts = count_images(TRAIN_DIR)
val_counts   = count_images(VAL_DIR)
test_counts  = count_images(TEST_DIR)

df_counts = pd.DataFrame({'Train': train_counts, 'Validation': val_counts, 'Test': test_counts})
print(df_counts)
print(f'\nTotal training images: {sum(train_counts.values())}')

# Plot class distribution
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
colors = ['#4A90D9', '#E74C3C', '#2ECC71', '#F39C12']

for ax, (split, counts) in zip(axes, [('Train', train_counts), ('Validation', val_counts), ('Test', test_counts)]):
    bars = ax.bar(counts.keys(), counts.values(), color=colors, edgecolor='black', linewidth=0.5)
    ax.set_title(f'{split} Set Distribution', fontsize=14, fontweight='bold')
    ax.set_xlabel('Class', fontsize=11)
    ax.set_ylabel('Image Count', fontsize=11)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, counts.values()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle("Alzheimer's MRI Dataset — Class Distribution", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Visualize Sample MRI Images

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

idx = 0
for cls in CLASS_NAMES:
    cls_path = os.path.join(TRAIN_DIR, cls)
    images = os.listdir(cls_path)[:2]
    for img_name in images:
        img = load_img(os.path.join(cls_path, img_name), target_size=IMG_SIZE)
        axes[idx].imshow(img, cmap='gray')
        axes[idx].set_title(cls, fontsize=11, fontweight='bold')
        axes[idx].axis('off')
        idx += 1

plt.suptitle("Sample MRI Brain Scans — All 4 Classes", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('sample_mri_images.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Data Preprocessing & Augmentation

In [ ]:
# Training augmentation — helps model generalize better
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    zoom_range=0.1,
    brightness_range=[0.9, 1.1],
    fill_mode='nearest'
)

# Validation and test — only rescale, no augmentation
val_test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    TRAIN_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=True
)

val_gen = val_test_datagen.flow_from_directory(
    VAL_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

test_gen = val_test_datagen.flow_from_directory(
    TEST_DIR, target_size=IMG_SIZE, batch_size=BATCH_SIZE,
    class_mode='categorical', shuffle=False
)

CLASS_INDICES = train_gen.class_indices
print('Class indices:', CLASS_INDICES)

## 6. Build CNN Model — EfficientNetB0 Transfer Learning

In [ ]:
def build_model(num_classes):
    # Load EfficientNetB0 pretrained on ImageNet — freeze base layers
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False  # freeze during initial training

    # Custom classification head
    inputs  = keras.Input(shape=(224, 224, 3))
    x       = base_model(inputs, training=False)
    x       = layers.GlobalAveragePooling2D()(x)
    x       = layers.BatchNormalization()(x)
    x       = layers.Dense(256, activation='relu')(x)
    x       = layers.Dropout(0.4)(x)
    x       = layers.Dense(128, activation='relu')(x)
    x       = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = Model(inputs, outputs)
    return model, base_model

model, base_model = build_model(NUM_CLASSES)

model.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 7. Train the Model — Phase 1 (Frozen Base)

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_accuracy', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_model.keras', monitor='val_accuracy', save_best_only=True, verbose=1)
]

print('Phase 1: Training classification head (base frozen)...')
history1 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)
print('Phase 1 training complete!')

## 8. Fine-Tuning — Phase 2 (Unfreeze Top Layers)

In [ ]:
# Unfreeze top 30 layers of EfficientNet for fine-tuning
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=Adam(learning_rate=1e-5),   # very low LR for fine-tuning
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print('Phase 2: Fine-tuning top layers...')
history2 = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)
print('Fine-tuning complete!')

## 9. Training Curves — Accuracy & Loss

In [ ]:
# Combine both phases
all_acc     = history1.history['accuracy']     + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss    = history1.history['loss']         + history2.history['loss']
all_val_loss= history1.history['val_loss']     + history2.history['val_loss']
phase1_end  = len(history1.history['accuracy'])

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Accuracy
axes[0].plot(all_acc,     label='Train Accuracy',      color='#2ECC71', linewidth=2)
axes[0].plot(all_val_acc, label='Validation Accuracy', color='#E74C3C', linewidth=2)
axes[0].axvline(x=phase1_end-1, color='gray', linestyle='--', label='Fine-tuning starts')
axes[0].set_title('Model Accuracy Over Epochs', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Loss
axes[1].plot(all_loss,     label='Train Loss',      color='#3498DB', linewidth=2)
axes[1].plot(all_val_loss, label='Validation Loss', color='#E67E22', linewidth=2)
axes[1].axvline(x=phase1_end-1, color='gray', linestyle='--', label='Fine-tuning starts')
axes[1].set_title('Model Loss Over Epochs', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle("Training History — Phase 1 + Fine-tuning", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Evaluate on Test Set

In [ ]:
# Load best saved model
model = keras.models.load_model('best_model.keras')

test_loss, test_acc = model.evaluate(test_gen, verbose=1)
print(f'\nTest Accuracy : {test_acc*100:.2f}%')
print(f'Test Loss     : {test_loss:.4f}')

## 11. Confusion Matrix

In [ ]:
test_gen.reset()
y_pred_probs = model.predict(test_gen, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_gen.classes
labels = list(CLASS_INDICES.keys())

cm_matrix = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(
    cm_matrix, annot=True, fmt='d', cmap='Blues',
    xticklabels=labels, yticklabels=labels,
    linewidths=0.5, linecolor='gray',
    annot_kws={'size': 13, 'weight': 'bold'}
)
ax.set_xlabel('Predicted Label', fontsize=13)
ax.set_ylabel('True Label', fontsize=13)
ax.set_title("Confusion Matrix — Test Set", fontsize=15, fontweight='bold')
plt.xticks(rotation=20, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report:')
print(classification_report(y_true, y_pred, target_names=labels))

## 12. Per-Class Accuracy Bar Chart

In [ ]:
per_class_acc = cm_matrix.diagonal() / cm_matrix.sum(axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
colors_acc = ['#2ECC71' if a >= 0.8 else '#E67E22' if a >= 0.6 else '#E74C3C' for a in per_class_acc]
bars = ax.bar(labels, per_class_acc * 100, color=colors_acc, edgecolor='black', linewidth=0.5)
ax.set_ylim(0, 115)
ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Per-Class Accuracy on Test Set', fontsize=14, fontweight='bold')
ax.axhline(y=80, color='gray', linestyle='--', alpha=0.6, label='80% threshold')
ax.legend()
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
            f'{acc*100:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('per_class_accuracy.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. ⭐ Grad-CAM Visualization — What Does the Model See?
Grad-CAM highlights which brain regions activated the model's decision — this is the unique explainability feature of this project.

In [ ]:
def get_gradcam_heatmap(model, img_array, last_conv_layer_name='top_conv'):
    """Generate Grad-CAM heatmap for a given image."""
    grad_model = Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        pred_index = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]

    grads = tape.gradient(class_channel, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), pred_index.numpy(), predictions[0].numpy()


def overlay_gradcam(img_path, heatmap, alpha=0.45):
    """Overlay Grad-CAM heatmap on original MRI."""
    img = load_img(img_path, target_size=IMG_SIZE)
    img_array = img_to_array(img)
    heatmap_resized = np.uint8(255 * heatmap)
    jet = cm.get_cmap('jet')
    jet_colors = jet(np.arange(256))[:, :3]
    jet_heatmap = jet_colors[heatmap_resized]
    jet_heatmap = keras.utils.array_to_img(jet_heatmap)
    jet_heatmap = jet_heatmap.resize(IMG_SIZE)
    jet_heatmap = img_to_array(jet_heatmap)
    superimposed = jet_heatmap * alpha + img_array
    superimposed = keras.utils.array_to_img(superimposed)
    return np.array(img), np.array(superimposed)


# Generate Grad-CAM for one sample from each class
fig, axes = plt.subplots(4, 2, figsize=(10, 18))
idx_map = {v: k for k, v in CLASS_INDICES.items()}

for row, cls in enumerate(CLASS_NAMES):
    cls_path = os.path.join(TEST_DIR, cls)
    img_file = os.listdir(cls_path)[0]
    img_path = os.path.join(cls_path, img_file)

    img_array = img_to_array(load_img(img_path, target_size=IMG_SIZE)) / 255.0
    img_array = np.expand_dims(img_array, axis=0)

    heatmap, pred_idx, probs = get_gradcam_heatmap(model, img_array)
    original, overlaid = overlay_gradcam(img_path, heatmap)

    pred_label = idx_map[pred_idx]
    confidence = probs[pred_idx] * 100
    correct = '✓' if pred_label == cls else '✗'

    axes[row, 0].imshow(original, cmap='gray')
    axes[row, 0].set_title(f'Original — True: {cls}', fontsize=10, fontweight='bold')
    axes[row, 0].axis('off')

    axes[row, 1].imshow(overlaid)
    axes[row, 1].set_title(f'Grad-CAM | Pred: {pred_label} ({confidence:.1f}%) {correct}',
                           fontsize=10, fontweight='bold',
                           color='green' if correct == '✓' else 'red')
    axes[row, 1].axis('off')

plt.suptitle("Grad-CAM: Brain Regions Driving Model Predictions",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gradcam_visualizations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Grad-CAM visualization complete!')

## 14. Final Summary

In [ ]:
print('=' * 55)
print("  ALZHEIMER'S MRI STAGE CLASSIFIER — FINAL RESULTS")
print('=' * 55)
print(f'  Model         : EfficientNetB0 (Transfer Learning)')
print(f'  Classes       : {NUM_CLASSES}')
print(f'  Test Accuracy : {test_acc*100:.2f}%')
print(f'  Test Loss     : {test_loss:.4f}')
print()
print('  Per-Class Accuracy:')
for cls, acc in zip(labels, per_class_acc):
    print(f'    {cls:<22}: {acc*100:.1f}%')
print()
print('  Outputs saved:')
print('    - best_model.keras')
print('    - class_distribution.png')
print('    - sample_mri_images.png')
print('    - training_curves.png')
print('    - confusion_matrix.png')
print('    - per_class_accuracy.png')
print('    - gradcam_visualizations.png')
print('=' * 55)